# Парсинг

Источник данных

Сайт: https://mel.fm/zhizn (раздел «Образ жизни»)

Корпус собран вручную через скроллинг страницы в браузере с последующим сохранением HTML и извлечением статей.

Разметка выполнена автоматически на основе структуры сайта. На mel.fm каждая статья уже принадлежит одному из подразделов:

| Подраздел | Рубрика |
|-----------|---------|
| /razbor | тренды |
| /istorii | истории |
| /knigi | культура |
| /razvlecheniya | хобби |
| /retsepty | рецепты |

При парсинге HTML извлекался класс .tag, содержащий название рубрики.

In [ ]:
from google.colab import files
uploaded = files.upload()  # выбери файл mel_zhizn_full.html

from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

with open("mel_zhizn_full.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

data = []
seen_urls = set()

cards = soup.find_all("div", class_="tile-card")
print(f"Найдено карточек: {len(cards)}")

for card in cards:
    if len(data) >= 550:
        break

    link_tag = card.find("a", href=True)
    if not link_tag:
        continue

    article_url = urljoin("https://mel.fm", link_tag["href"])

    if article_url in seen_urls:
        continue
    seen_urls.add(article_url)

    # Определяем рубрику
    tag_div = card.find("div", class_="tag")
    rubric_text = tag_div.text.strip() if tag_div else ""

    rubric_map = {
        "Тренды": "тренды",
        "Истории": "истории",
        "Культура": "культура",
        "Хобби": "хобби",
        "Рецепты": "рецепты"
    }

    rubric = rubric_map.get(rubric_text)
    if not rubric:
        continue

    print(f"Обработка: {article_url}")

    # Текст статьи нужно догрузить отдельно
    import requests
    import time

    time.sleep(0.3)
    resp = requests.get(article_url, headers={"User-Agent": "Mozilla/5.0"})

    if resp.status_code == 200:
        art_soup = BeautifulSoup(resp.text, "html.parser")
        paragraphs = art_soup.find_all("p")
        text = " ".join([p.text.strip() for p in paragraphs if len(p.text.strip()) > 20])

        if len(text) >= 200:
            data.append({
                "text": text,
                "label": rubric
            })
            print(f"  {len(data)}/550: сохранено")

df = pd.DataFrame(data)
df.to_csv("mel_500.csv", index=False, encoding="utf-8")
print(f"\nСобрано {len(data)} примеров")
print(df["label"].value_counts())

Saving mel_zhizn_full.html to mel_zhizn_full (1).html
Найдено карточек: 945
Обработка: https://mel.fm/zhizn/knigi/8913056-ya-byl-kak-strashila-kotoromu-vstavili-mozgi-arkhitektor-oleg-karlson--o-knigakh-i-ikh-vliyanii-na-n
  1/550: сохранено
Обработка: https://mel.fm/zhizn/knigi/6892405-festival-kotov-okeansky-subbotnik-i-klassika-v-malom-teatre-kuda-poyti-s-detmi-vo-vtoroy-polovine-iy
  2/550: сохранено
Обработка: https://mel.fm/zhizn/retsepty/1953476-retsept-cheremshi-za-20-minut-dachnaya-zagotovka-iz-chesnochnykh-strelok
  3/550: сохранено
Обработка: https://mel.fm/zhizn/razbor/8621435-you-chuvak-sharish-za-politiku
  4/550: сохранено
Обработка: https://mel.fm/zhizn/knigi/3147085-zachem-brat-detey-na-piknik-afishi-v-moskve-gid-po-festivalyu-dlya-roditeley
  5/550: сохранено
Обработка: https://mel.fm/zhizn/istorii/5940273-glyadya-na-detey-ya-dumala-tolko-o-smerti-pisatelnitsa-mar-garsia-puch--o-poslerodovoy-depressii-i-m
  6/550: сохранено
Обработка: https://mel.fm/zhizn/knigi/307941

# Загрузка данных

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("mel_500.csv")
print(f"Загружено {len(df)} примеров")
print(df["label"].value_counts())
df.head()

Загружено 550 примеров
label
культура    162
хобби       107
истории      99
рецепты      92
тренды       90
Name: count, dtype: int64


,text,label
0,"«Я был как Страшила, которому вставили мозги»:...",культура
1,"Фестиваль котов, океанский субботник и классик...",культура
2,Рецепт чесночных стрелок за 20 минут: маринуем...,рецепты
3,"«Йоу, чувак, шаришь за Госдуму?»: как политиче...",тренды
4,Зачем брать детей на Пикник Афиши в Москве: ги...,культура


# Предобработка

In [ ]:
# Проверяем пропуски
df = df[["text", "label"]].dropna()

# Преобразуем текстовые метки в категории
df["label"] = df["label"].astype("category")

# Создаём числовые метки
df["label_id"] = df["label"].cat.codes

print("Соответствие меток и чисел:")
for idx, name in enumerate(df["label"].cat.categories):
    print(f"  {idx} -> {name}")

df.head()

Соответствие меток и чисел:
  0 -> истории
  1 -> культура
  2 -> рецепты
  3 -> тренды
  4 -> хобби


,text,label,label_id
0,"«Я был как Страшила, которому вставили мозги»:...",культура,1
1,"Фестиваль котов, океанский субботник и классик...",культура,1
2,Рецепт чесночных стрелок за 20 минут: маринуем...,рецепты,2
3,"«Йоу, чувак, шаришь за Госдуму?»: как политиче...",тренды,3
4,Зачем брать детей на Пикник Афиши в Москве: ги...,культура,1


# Векторизация

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["text"]).toarray()
y = df["label_id"].values

print(f"Матрица признаков X: {X.shape}")
print(f"Вектор меток y: {y.shape}")

Матрица признаков X: (550, 5000)
Вектор меток y: (550,)


# Разделение выборок

In [ ]:
from sklearn.model_selection import train_test_split

# Разделяем на обучающую (70%), валидационную (15%), тестовую (15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Обучающая выборка: {X_train.shape[0]} примеров")
print(f"Валидационная выборка: {X_val.shape[0]} примеров")
print(f"Тестовая выборка: {X_test.shape[0]} примеров")

Обучающая выборка: 385 примеров
Валидационная выборка: 82 примеров
Тестовая выборка: 83 примеров


# Тензоры и DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Тензоры для каждой выборки
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

print("Готово")

Готово


# Модель CNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * (input_dim // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = CNN1D(X.shape[1], len(df["label"].unique()))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Модель создана")
print(f"Входной размер: {X.shape[1]}")
print(f"Количество классов: {len(df['label'].unique())}")

Модель создана
Входной размер: 5000
Количество классов: 5


Обоснование гиперпараметров

Все гиперпараметры подбирались исходя из размера данных (550 примеров) и задачи (классификация на 5 рубрик).

| Параметр | Выбранное значение | Почему так |
|----------|-------------------|-------------|
| **max_features** | 5000 самых частотных слов | Этого достаточно для новостных текстов. Меньше — потеряем смысл, больше — модель будет переобучаться. |
| **kernel_size** | 3 слова | Лучше всего ловит устойчивые сочетания из 3 слов (триграммы). Для русского языка стандартный выбор. |
| **filters** | 32 → 64 | Увеличиваем сложность паттернов: сначала простые, потом более сложные. |
| **pool_size** | 2 | MaxPooling с окном 2 уменьшает размер признаков в 2 раза, оставляя самое важное. |
| **batch_size** | 32 | Классическое значение для небольших датасетов — и стабильно, и не слишком долго. |
| **lr** | 0.001 | Значение по умолчанию для оптимизатора Adam. Даёт плавную сходимость без скачков. |
| **epochs** | 5 | Этого хватило, так как качество на валидации перестало расти. Дольше — только переобучение. |

Замечание: в этой модели нет слоя Embedding. Вместо этого мы используем CountVectorizer (мешок слов) и подаём разреженные векторы напрямую в свёртку.

# Обучение

In [ ]:
for epoch in range(5):
    # Обучение
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Валидация
    model.eval()
    val_loss = 0
    correct = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()

    val_acc = correct / len(val_dataset)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.4f}")

Epoch 1, Loss: 0.0325, Val Loss: 0.0209, Val Acc: 1.0000
Epoch 2, Loss: 0.0193, Val Loss: 0.0126, Val Acc: 1.0000
Epoch 3, Loss: 0.0116, Val Loss: 0.0105, Val Acc: 1.0000
Epoch 4, Loss: 0.0070, Val Loss: 0.0084, Val Acc: 1.0000
Epoch 5, Loss: 0.0041, Val Loss: 0.0064, Val Acc: 1.0000


# Метрики

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)

acc = accuracy_score(y_test, predicted)
p, r, f1, _ = precision_recall_fscore_support(y_test, predicted, average="weighted")

print("="*50)
print("МЕТРИКИ КАЧЕСТВА НА ТЕСТОВОЙ ВЫБОРКЕ")
print("="*50)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
print(f"F1-score:  {f1:.4f}")
print("="*50)

МЕТРИКИ КАЧЕСТВА НА ТЕСТОВОЙ ВЫБОРКЕ
Accuracy:  0.9759
Precision: 0.9786
Recall:    0.9759
F1-score:  0.9762


# Детальный отчёт

In [ ]:
from sklearn.metrics import classification_report

# Словарь для преобразования чисел обратно в названия
id_to_label = {idx: name for idx, name in enumerate(df["label"].cat.categories)}

print("Детальный отчёт по каждой рубрике (на тестовой выборке):")
print(classification_report(y_test, predicted, target_names=list(id_to_label.values())))

Детальный отчёт по каждой рубрике (на тестовой выборке):
              precision    recall  f1-score   support

     истории       1.00      0.93      0.97        15
    культура       1.00      1.00      1.00        24
     рецепты       1.00      1.00      1.00        14
      тренды       1.00      0.93      0.96        14
       хобби       0.89      1.00      0.94        16

    accuracy                           0.98        83
   macro avg       0.98      0.97      0.97        83
weighted avg       0.98      0.98      0.98        83



**Анализ результатов**

Accuracy 97.6% — отличный результат для 5 классов на относительно небольшой выборке.

Идеальное качество на двух классах:

Культура — 100% (precision, recall, f1-score = 1.0)
Рецепты — 100%

Стабильное обучение без переобучения:

Train Loss: 0.032 → 0.004 (уменьшается)
Val Loss: 0.021 → 0.006 (тоже уменьшается)
Разрыв между train и val небольшой — значит, модель обобщает, а не просто запоминает.

**Что можно улучшить**

Класс «хобби» (F1 = 0.94):

Precision = 0.89 — хуже всего среди всех классов
Что это значит: 2 статьи были ошибочно отнесены к хобби, хотя относятся к другому классу.
Почему так: хобби — самая размытая категория (игры, рукоделие, спорт, коллекционирование). Она пересекается с трендами и историями.

Классы «истории» и «тренды» (Recall = 0.93):

По 1 статье из каждого класса модель не узнала (пропустила)
Вероятно, эти статьи были короткими или использовали нехарактерную лексику.